# Variables de planta: razones entre áreas

Este notebook decide y construye las razones que comparan el frío con el resto de la planta y la planta con su producción. Cada número que aparece en las tablas sale de una celda que se puede volver a ejecutar, y toda decisión se mide solo con `X_train`, nunca con validación ni prueba. Las justificaciones resumidas están en el ADR 0013.

## 1. Preparación: tabla limpia y división

Se carga la tabla completa, se divide en entrenamiento, validación y prueba como en los otros notebooks, se aprenden las reglas de limpieza solo con entrenamiento y se aplican a toda la tabla. El control debe coincidir con el del notebook de preprocesamiento: 1.212 días y 276 columnas.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple
from preprocessing_pipeline import cargar_tabla_completa, dividir_por_fecha
from limpieza import aprender_reglas_de_limpieza, limpiar
from variables_planta import (aprender_reglas_de_planta, crear_variables_planta, RAZONES, NOMBRES_NUEVAS,
                              DENOMINADORES, FRACCION_DE_LA_MEDIANA, HL_ENVASADOS, PLANTA_KW)

tabla_completa = cargar_tabla_completa(Path("../data"))
X_train, X_val, X_test = dividir_por_fecha(tabla_completa)
reglas_limpieza = aprender_reglas_de_limpieza(X_train)            # reglas medidas solo con entrenamiento
tabla_limpia = limpiar(tabla_completa, reglas_limpieza).copy()
train_limpio, val_limpio, test_limpio = dividir_por_fecha(tabla_limpia)

control = pd.DataFrame({"Valor": {
    "Días (filas) de la tabla limpia": len(tabla_limpia),
    "Columnas de la tabla limpia": tabla_limpia.shape[1],
    "Días de entrenamiento": len(train_limpio),
    "Días de validación": len(val_limpio),
    "Días de prueba": len(test_limpio)}}).rename_axis("Control").reset_index()
print("Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas)")
display(tabla_simple(control, precision=0))

Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas)


,Control,Valor
0,Días (filas) de la tabla limpia,1212
1,Columnas de la tabla limpia,276
2,Días de entrenamiento,848
3,Días de validación,182
4,Días de prueba,182


## 2. Las razones y sus denominadores

Se crean seis razones con sentido físico, todas con mediciones del mismo día y en sus unidades reales: el peso del frío y de los servicios dentro de la planta, y cuánta energía, aire y gas se gasta por cada hectolitro envasado. La tabla muestra qué divide cada una.

In [2]:
razones = pd.DataFrame(RAZONES, columns=["Variable nueva", "Numerador", "Denominador"])
print("Tabla 2. Las seis razones: qué se divide por qué")
display(tabla_simple(razones))

Tabla 2. Las seis razones: qué se divide por qué


,Variable nueva,Numerador,Denominador
0,frio_sobre_planta,Frio (Kw),Planta (Kw)
1,frio_por_hl_envasado,Frio (Kw),Consolidado Produccion | Hl Cerveza Envasada
2,planta_por_hl_envasado,Planta (Kw),Consolidado Produccion | Hl Cerveza Envasada
3,aire_por_hl_envasado,Consolidado Aire | Aire Producido (M3),Consolidado Produccion | Hl Cerveza Envasada
4,gas_por_hl_envasado,Consolidado GasVapor | Gas Planta (Mj),Consolidado Produccion | Hl Cerveza Envasada
5,servicios_sobre_planta,Servicios (Kw),Planta (Kw)


## 3. Qué pasa con los denominadores chicos

Dividir por una producción de cero hectolitros no se puede, y dividir por una producción casi nula da un número enorme que no describe el consumo de un día normal. La primera tabla cuenta, solo en entrenamiento, cuántos días tiene cada denominador en cero, vacío o por debajo de un umbral. El umbral propuesto es una fracción fija de la mediana de entrenamiento (la constante `FRACCION_DE_LA_MEDIANA`); la columna «Menor valor mayor que cero» muestra qué tan chico puede llegar a ser un denominador que no es cero.

In [3]:
reglas_planta = aprender_reglas_de_planta(train_limpio)
filas = []
for columna in DENOMINADORES:
    valores = pd.to_numeric(train_limpio[columna], errors="coerce")
    umbral = reglas_planta["umbral_denominador"][columna]
    filas.append({"Denominador": columna, "Mediana": valores.median(), "Umbral": umbral,
                  "Días en cero": int((valores == 0).sum()), "Días vacíos": int(valores.isna().sum()),
                  "Días bajo el umbral (incluye los ceros)": int((valores < umbral).sum()),
                  "Menor valor mayor que cero": valores[valores > 0].min()})
print(f"Tabla 3. Denominadores en entrenamiento: umbral = {FRACCION_DE_LA_MEDIANA:.0%} de la mediana ({len(train_limpio)} días)")
display(tabla_simple(pd.DataFrame(filas), precision=0))

Tabla 3. Denominadores en entrenamiento: umbral = 10% de la mediana (848 días)

,Denominador,Mediana,Umbral,Días en cero,Días vacíos,Días bajo el umbral (incluye los ceros),Menor valor mayor que cero
0,Consolidado Produccion | Hl Cerveza Envasada,9798,980,27,2,41,26
1,Planta (Kw),70059,7006,0,18,0,10770


La segunda tabla compara la razón del frío por hectolitro envasado con y sin la regla. «Sin la regla» solo evita el cero (da vacío) pero acepta cualquier otro denominador; «con la regla» deja vacíos también los días bajo el umbral.

In [4]:
frio = pd.to_numeric(train_limpio["Frio (Kw)"], errors="coerce")
hl = pd.to_numeric(train_limpio[HL_ENVASADOS], errors="coerce")
sin_regla = frio / hl.where(hl > 0)
con_regla = frio / hl.where(hl >= reglas_planta["umbral_denominador"][HL_ENVASADOS])
comparacion = pd.DataFrame({
    "Sin la regla": [sin_regla.notna().sum(), sin_regla.median(), sin_regla.quantile(0.99), sin_regla.max()],
    "Con la regla": [con_regla.notna().sum(), con_regla.median(), con_regla.quantile(0.99), con_regla.max()]},
    index=["Días con valor", "Mediana (kW por Hl)", "Percentil 99 (kW por Hl)", "Máximo (kW por Hl)"]).rename_axis("Frío por hectolitro envasado").reset_index()
print("Tabla 4. Frío por hectolitro envasado, con y sin la regla del denominador (solo entrenamiento)")
display(tabla_simple(comparacion, precision=0))

Tabla 4. Frío por hectolitro envasado, con y sin la regla del denominador (solo entrenamiento)


,Frío por hectolitro envasado,Sin la regla,Con la regla
0,Días con valor,805,791
1,Mediana (kW por Hl),3,3
2,Percentil 99 (kW por Hl),97,6
3,Máximo (kW por Hl),871,13


La regla recorta los extremos: sin ella, unos pocos días de producción casi nula hacen que el percentil 99 y el máximo queden muy por encima de lo que da un día normal, mientras que la mediana casi no cambia. A cambio se pierden pocos días con valor (la diferencia en la primera fila). La planta en kW no tiene días en cero ni bajo el umbral, de modo que las razones que la dividen solo quedan vacías en los días inválidos que ya anuló la limpieza.

## 4. Construcción y controles

Las reglas se aprenden con el tramo de entrenamiento y las razones se calculan sobre la tabla limpia completa. La primera tabla cuenta cuántos días quedan vacíos en cada razón y en cada tramo; la segunda resume sus valores en entrenamiento.

In [5]:
tabla_planta = crear_variables_planta(tabla_limpia, reglas_planta)
tp_train, tp_val, tp_test = dividir_por_fecha(tabla_planta)

vacios = pd.DataFrame({"Entrenamiento": tp_train[NOMBRES_NUEVAS].isna().sum(),
                       "Validación": tp_val[NOMBRES_NUEVAS].isna().sum(),
                       "Prueba": tp_test[NOMBRES_NUEVAS].isna().sum()}).rename_axis("Variable").reset_index()
print("Tabla 5. Días vacíos de cada razón por tramo (entrenamiento 848 días, validación 182, prueba 182)")
display(tabla_simple(vacios, precision=0))

resumen = tp_train[NOMBRES_NUEVAS].describe().T[["min", "25%", "50%", "75%", "max"]].rename_axis("Variable").reset_index()
print("Tabla 6. Valores de cada razón en entrenamiento")
display(tabla_simple(resumen, precision=3))

Tabla 5. Días vacíos de cada razón por tramo (entrenamiento 848 días, validación 182, prueba 182)


,Variable,Entrenamiento,Validación,Prueba
0,frio_sobre_planta,18,5,3
1,frio_por_hl_envasado,57,43,38
2,planta_por_hl_envasado,58,43,38
3,aire_por_hl_envasado,43,41,37
4,gas_por_hl_envasado,43,41,37
5,servicios_sobre_planta,18,5,3


Tabla 6. Valores de cada razón en entrenamiento


,Variable,min,25%,50%,75%,max
0,frio_sobre_planta,0.023,0.335,0.375,0.406,0.962
1,frio_por_hl_envasado,0.274,2.105,2.621,3.322,12.631
2,planta_por_hl_envasado,3.864,6.091,7.071,8.617,30.646
3,aire_por_hl_envasado,0.644,4.991,5.823,7.441,38.100
4,gas_por_hl_envasado,27.961,51.395,59.426,73.036,404.166
5,servicios_sobre_planta,0.154,0.583,0.611,0.646,1.191


In [6]:
controles = {
    "Mismas filas e índice que la tabla limpia": tabla_planta.index.equals(tabla_limpia.index),
    "Las columnas originales no cambian": tabla_planta[tabla_limpia.columns].equals(tabla_limpia),
    "Solo se agregan las seis razones, al final": list(tabla_planta.columns) == list(tabla_limpia.columns) + NOMBRES_NUEVAS,
    "Ninguna razón vale infinito": not np.isinf(tabla_planta[NOMBRES_NUEVAS].to_numpy(dtype=float)).any(),
    "Las reglas salen solo de entrenamiento (recalcular da lo mismo)": aprender_reglas_de_planta(tabla_limpia.loc[train_limpio.index]) == reglas_planta,
}
fila = train_limpio.index[train_limpio["Frio (Kw)"].notna() & train_limpio[PLANTA_KW].notna()][0]
controles["Una fila al azar: frío sobre planta = frío / planta"] = bool(np.isclose(
    tabla_planta.loc[fila, "frio_sobre_planta"], tabla_limpia.loc[fila, "Frio (Kw)"] / tabla_limpia.loc[fila, PLANTA_KW]))
print("Tabla 7. Controles (todos deben dar True)")
display(tabla_simple(pd.DataFrame({"Resultado": controles}).rename_axis("Control").reset_index()))

Tabla 7. Controles (todos deben dar True)


,Control,Resultado
0,Mismas filas e índice que la tabla limpia,True
1,Las columnas originales no cambian,True
2,"Solo se agregan las seis razones, al final",True
3,Ninguna razón vale infinito,True
4,Las reglas salen solo de entrenamiento (recalcular da lo mismo),True
5,Una fila al azar: frío sobre planta = frío / planta,True


Las razones que dividen por los hectolitros envasados tienen más días vacíos que las que dividen por el consumo de la planta. El motivo se debe a que en un día en que la planta casi no envasó (cero hectolitros o muy pocos), la regla de la sección 3 no permite dividir y esa razón queda vacía ese día. El consumo de la planta, en cambio, casi nunca es cero, así que las otras razones casi no tienen vacíos.

La tabla 5 muestra otra diferencia: en validación y prueba, esos vacíos son una parte mucho mayor de los días que en entrenamiento. La tabla 8 lo explica: cuenta, en cada tramo, los días con producción envasada por debajo del umbral que se aprendió en entrenamiento, y hay proporcionalmente más de esos días en validación y prueba. La regla se aplica igual en los tres tramos; lo que cambia es cuántos días de producción casi nula tiene cada uno. No sabemos la causa (podrían ser paradas de la planta, una temporada de menor producción o un problema de registro) y estas tablas no permiten distinguirlas.

Cuando se completen esos vacíos en el Pipeline, hay que decidir cómo hacerlo, porque un día con la planta casi parada no debería recibir el valor de un día normal.

In [7]:
umbral_hl = reglas_planta["umbral_denominador"][HL_ENVASADOS]
filas = []
for tramo, datos in [("Entrenamiento", train_limpio), ("Validación", val_limpio), ("Prueba", test_limpio)]:
    hl_tramo = pd.to_numeric(datos[HL_ENVASADOS], errors="coerce")
    filas.append({"Tramo": tramo, "Días": len(datos), "Días en cero": int((hl_tramo == 0).sum()),
                  "Días bajo el umbral (incluye los ceros)": int((hl_tramo < umbral_hl).sum()),
                  "Porcentaje de días bajo el umbral": 100 * (hl_tramo < umbral_hl).mean(),
                  "Mediana de hectolitros envasados": hl_tramo.median()})
print("Tabla 8. Días de producción envasada casi nula por tramo (umbral aprendido en entrenamiento)")
display(tabla_simple(pd.DataFrame(filas), precision=1))

Tabla 8. Días de producción envasada casi nula por tramo (umbral aprendido en entrenamiento)


,Tramo,Días,Días en cero,Días bajo el umbral (incluye los ceros),Porcentaje de días bajo el umbral,Mediana de hectolitros envasados
0,Entrenamiento,848,27,41,4.8,9797.5
1,Validación,182,27,38,20.9,6721.0
2,Prueba,182,17,36,19.8,9402.0


## 5. Cuánto se parecen al frío de mañana

Esta tabla mide, en entrenamiento, el parecido de Spearman de cada razón con el frío de mañana (la fila siguiente), y lo compara con el del frío de hoy. Es solo una medida descriptiva: una razón puede aportar aunque se parezca poco al objetivo, y la selección de la etapa siguiente decide con qué variables se queda el modelo. La última columna indica cuánto se parece cada razón al frío de hoy, para ver si es una copia de él.

In [8]:
objetivo = train_limpio["objetivo"]
def spearman(x, y):
    pares = pd.DataFrame({"x": x, "y": y}).dropna()
    return pares["x"].corr(pares["y"], method="spearman"), len(pares)

frio_hoy = tp_train["Frio (Kw)"]
filas = [{"Variable": "Frío de hoy (referencia)", "Spearman con el frío de mañana": spearman(frio_hoy, objetivo)[0],
          "Días usados": spearman(frio_hoy, objetivo)[1], "Spearman con el frío de hoy": 1.0}]
for nombre in NOMBRES_NUEVAS:
    r, n = spearman(tp_train[nombre], objetivo)
    filas.append({"Variable": nombre, "Spearman con el frío de mañana": r, "Días usados": n,
                  "Spearman con el frío de hoy": spearman(tp_train[nombre], frio_hoy)[0]})
print("Tabla 9. Parecido de cada razón con el frío de mañana (solo entrenamiento)")
display(tabla_simple(pd.DataFrame(filas), precision=3))

Tabla 9. Parecido de cada razón con el frío de mañana (solo entrenamiento)


,Variable,Spearman con el frío de mañana,Días usados,Spearman con el frío de hoy
0,Frío de hoy (referencia),0.847,817,1.000
1,frio_sobre_planta,0.553,816,0.666
2,frio_por_hl_envasado,0.317,778,0.421
3,planta_por_hl_envasado,0.044,777,0.103
4,aire_por_hl_envasado,-0.223,790,-0.209
5,gas_por_hl_envasado,-0.409,790,-0.428
6,servicios_sobre_planta,0.128,816,0.183


Ninguna razón se parece al frío de mañana tanto como el frío de hoy, y ninguna es una copia de él. La participación del frío en la planta es la más parecida, y la intensidad de la planta por hectolitro es casi independiente del objetivo. Las razones de aire y de gas por hectolitro tienen signo negativo: valen más cuando el frío de mañana es menor. Esto es descriptivo y no explica la causa; la selección de la etapa siguiente decide cuáles aportan algo más allá del frío de hoy.

## 6. Lo que se evaluó y no se incluye

**Compresores Mycom sobre el frío.** La suma de los compresores Mycom se parece mucho al frío del mismo día, así que dividir una por otro daría casi una constante. La primera tabla mide ese parecido y cuánto varía la razón (tabla 10).

**Razones por línea.** Las líneas 2, 3, 4 y 5 tienen su propia producción en hectolitros. La segunda tabla (11) muestra cuántos días esa producción es cero o queda bajo el umbral de la misma regla, y cuántos días vacíos tiene el consumo eléctrico de la línea.

In [9]:
columnas_mycom = [c for c in train_limpio.columns if c.startswith("Totalizadores Energia | KW Mycom")]
suma_mycom = train_limpio[columnas_mycom].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=len(columnas_mycom))
razon_mycom = suma_mycom / frio.where(frio > 0)
mycom = pd.DataFrame({"Valor": {
    "Columnas Mycom que sobreviven a la limpieza": len(columnas_mycom),
    "Spearman de la suma con el frío del mismo día": spearman(suma_mycom, frio)[0],
    "Mediana de suma Mycom / frío": razon_mycom.median(),
    "Variación relativa de esa razón (desvío / promedio)": razon_mycom.std() / razon_mycom.mean()}}).rename_axis("Medida").reset_index()
print("Tabla 10. Compresores Mycom y frío (solo entrenamiento)")
display(tabla_simple(mycom, precision=3))

lineas = {"Línea 2": ("Linea 2 (Kw)", "Consolidado Produccion | Hl Cerveza L2"),
          "Línea 3": ("Linea 3 (Kw)", "Consolidado Produccion | Hl Cerveza L3"),
          "Línea 4": ("Linea 4 (Kw)", "Consolidado Produccion | Hl Cerveza L4"),
          "Línea 5": (None, "Consolidado Produccion | Hl Cerveza L5")}
filas = []
for linea, (columna_kw, columna_hl) in lineas.items():
    hl_linea = pd.to_numeric(train_limpio[columna_hl], errors="coerce")
    umbral = FRACCION_DE_LA_MEDIANA * hl_linea.median()
    filas.append({"Línea": linea, "Mediana de hectolitros": hl_linea.median(),
                  "Días en cero": int((hl_linea == 0).sum()),
                  "Días bajo el umbral (incluye los ceros)": int((hl_linea < umbral).sum()),
                  "Días vacíos del consumo eléctrico": (int(pd.to_numeric(train_limpio[columna_kw], errors="coerce").isna().sum())
                                                          if columna_kw else "no hay columna")})
print("Tabla 11. Producción de cada línea en entrenamiento (848 días)")
display(tabla_simple(pd.DataFrame(filas), precision=0))

Tabla 10. Compresores Mycom y frío (solo entrenamiento)


,Medida,Valor
0,Columnas Mycom que sobreviven a la limpieza,7.000
1,Spearman de la suma con el frío del mismo día,0.991
2,Mediana de suma Mycom / frío,0.732
3,Variación relativa de esa razón (desvío / promedio),0.102


Tabla 11. Producción de cada línea en entrenamiento (848 días)


,Línea,Mediana de hectolitros,Días en cero,Días bajo el umbral (incluye los ceros),Días vacíos del consumo eléctrico
0,Línea 2,1208,134,194,19
1,Línea 3,3872,182,188,17
2,Línea 4,5224,91,101,17
3,Línea 5,0,492,0,no hay columna


El parecido de la suma de Mycom con el frío del mismo día es casi total, aunque la razón entre ambos no es una constante exacta porque varía algo de un día a otro. Aun así, esa suma dice casi lo mismo que el frío de hoy y no agrega información distinta, así que no se crea; queda como pendiente evaluar, en la selección, si algún compresor individual aporta. Las líneas pasan muchos días sin producir (en la línea 5 la mediana de hectolitros es cero y no hay columna de consumo eléctrico propio), por lo que sus razones quedarían vacías en una parte grande de los días y no se incluyen.

## 7. Conclusión

Las seis razones quedan construidas con una regla única para los denominadores, aprendida solo con entrenamiento y aplicada igual a validación, prueba y datos nuevos. No se incluyen la razón de Mycom sobre el frío ni las razones por línea, por lo que muestran las tablas 10 y 11. Las razones se calculan en kW y unidades reales, sin transformar ni escalar (ADR 0010), y los vacíos se imputan más adelante dentro del Pipeline. Luego de esto, aplicamos PolynomialFeatures sobre un conjunto reducido de dimensiones y después hacemoos selección de variables.